In [6]:

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

# 1. Pobranie datasetu i losowanie 2000 próbek
housing = fetch_california_housing(as_frame=True)

df = housing.frame.sample(
    n=2000,
    random_state=42
).reset_index(drop=True)

print("Wymiary datasetu:", df.shape)


# 2. Obliczenie macierzy korelacji
C = np.corrcoef(df.values.T)


# 3. Funkcja ASCII heatmap
def ascii_heatmap(C, labels):

    def symbol(value):
        if value <= -0.7:
            return "##"
        elif value <= -0.3:
            return "--"
        elif value < 0.3:
            return ".."
        elif value < 0.7:
            return "++"
        else:
            return "XX"

    print("\nASCII HEATMAP:")
    print(" " * 15 + " ".join(f"{x:>10}" for x in labels))
    print("-" * (15 + 11 * len(labels)))

    for i, label in enumerate(labels):
        row = " ".join(
            f"{symbol(C[i, j]):>10}"
            for j in range(len(labels))
        )
        print(f"{label:>15} {row}")


# 4. Wyświetlenie heatmapy
ascii_heatmap(C, df.columns)


# 5. Usunięcie targetu MedHouseVal
target_idx = df.columns.get_loc("MedHouseVal")

C_features = C.copy()

C_features[target_idx, :] = np.nan
C_features[:, target_idx] = np.nan

# Usunięcie przekątnej
np.fill_diagonal(C_features, np.nan)


# 6. Pobranie par z górnego trójkąta macierzy
i, j = np.triu_indices_from(C_features, k=1)

# Wartości bezwzględne korelacji
values = np.abs(C_features[i, j])

# 7. TOP 3 pary cech
top_indices = np.argsort(values)[-3:][::-1]

print("\nTOP 3 PARY CECH O NAJWYŻSZEJ KORELACJI:\n")

for idx in top_indices:
    feature1 = df.columns[i[idx]]
    feature2 = df.columns[j[idx]]
    correlation = C[i[idx], j[idx]]

    print(f"{feature1} - {feature2}: {correlation:.4f}")


# 8. Interpretacja
print("\nINTERPRETACJA:")

for idx in top_indices:
    feature1 = df.columns[i[idx]]
    feature2 = df.columns[j[idx]]
    correlation = C[i[idx], j[idx]]

    direction = "dodatnia" if correlation > 0 else "ujemna"

    print(
        f"{feature1} i {feature2}: "
        f"korelacja {direction} ({correlation:.4f})."
    )

print("\nWniosek:")
print(
    "Niektóre cechy datasetu są ze sobą silnie powiązane. "
    "Może to mieć znaczenie podczas wyboru cech "
    "i budowania modeli uczenia maszynowego."
)


Wymiary datasetu: (2000, 9)

ASCII HEATMAP:
                   MedInc   HouseAge   AveRooms  AveBedrms Population   AveOccup   Latitude  Longitude MedHouseVal
------------------------------------------------------------------------------------------------------------------
         MedInc         XX         ..         ++         ..         ..         ..         ..         ..         ++
       HouseAge         ..         XX         ..         ..         --         ..         ..         ..         ..
       AveRooms         ++         ..         XX         XX         ..         ..         ..         ..         ..
      AveBedrms         ..         ..         XX         XX         ..         ..         ..         ..         ..
     Population         ..         --         ..         ..         XX         ..         ..         ..         ..
       AveOccup         ..         ..         ..         ..         ..         XX         ..         ..         ..
       Latitude         ..         .